# Random PCA and Permutation MAE/R2 statistics

In [1]:
import pandas as pd
from pathlib import Path


## Configuration

In [2]:
BASELINE_OUT_ROOT = "comp"
SPACE_NAME = "output_proj"

MODEL_RUN_CONFIGS = [
    {
        "label": "mistral",
        "model_name": "mistralai/Mistral-7B-v0.1",
        "tokenizer_path": "mistralai/Mistral-7B-v0.1/tokenizer",
        "full_pca_dim": 4096,
        "candidate_dims": [5, 142, 997, 2084, 3031, 3459, 3938, 4088, 4096],
    },
    {
        "label": "mixtral",
        "model_name": "mistralai/Mixtral-8x7B-v0.1",
        "tokenizer_path": "mistralai/Mixtral-8x7B-v0.1/tokenizer",
        "full_pca_dim": 4096,
        "candidate_dims": [8, 158, 1111, 2156, 3052, 3457, 3957, 4093, 4096],
    },
    {
        "label": "gpt-oss",
        "model_name": "gpt-oss",
        "tokenizer_path": "gpt-oss/tokenizer",
        "full_pca_dim": 2880,
        "candidate_dims": [6, 182, 466, 739, 1591, 2264, 2532, 2868, 2880],
    },
]

PERMUTATION_MODEL_CONFIGS = [
    {
        "label": "mistral",
        "baseline_model_name": "mistralai/Mistral-7B-v0.1",
        "permutation_model_name": "permutation/mistralai/Mistral-7B-v0.1_rand",
    },
    {
        "label": "mixtral",
        "baseline_model_name": "mistralai/Mixtral-8x7B-v0.1",
        "permutation_model_name": "permutation/mistralai/Mixtral-8x7B-v0.1_rand",
    },
    {
        "label": "gpt-oss",
        "baseline_model_name": "gpt-oss",
        "permutation_model_name": "permutation/gpt-oss_rand",
    },
]


## Helpers

In [3]:
def compute_curve_r2(baseline_values, control_values):
    baseline_values = pd.Series(baseline_values, dtype=float).to_numpy()
    control_values = pd.Series(control_values, dtype=float).to_numpy()
    ss_res = float(((control_values - baseline_values) ** 2).sum())
    ss_tot = float(((baseline_values - baseline_values.mean()) ** 2).sum())
    if ss_tot == 0.0:
        return float("nan")
    return 1.0 - ss_res / ss_tot


def require_existing_paths(paths):
    missing_paths = [str(path) for path in paths if not path.exists()]
    if missing_paths:
        return False, missing_paths
    return True, []


def align_metric_curve(baseline_df, control_df, value_col, *, control_suffix):
    aligned = (
        baseline_df[["pca_dim", value_col]]
        .merge(
            control_df[["pca_dim", value_col]],
            on="pca_dim",
            how="inner",
            suffixes=("_baseline", f"_{control_suffix}"),
        )
        .sort_values("pca_dim")
        .reset_index(drop=True)
    )
    if aligned.empty:
        raise ValueError(f"No overlapping pca_dim values for {value_col}.")
    return aligned


def compute_morph_script_joint_stats(baseline_morph, baseline_script, control_morph, control_script, *, control_suffix):
    morph_aligned = align_metric_curve(
        baseline_morph,
        control_morph,
        "global_mean",
        control_suffix=control_suffix,
    )
    script_aligned = align_metric_curve(
        baseline_script,
        control_script,
        "mean_H",
        control_suffix=control_suffix,
    )

    morph_baseline_values = morph_aligned["global_mean_baseline"].to_numpy(dtype=float)
    morph_control_values = morph_aligned[f"global_mean_{control_suffix}"].to_numpy(dtype=float)
    script_baseline_values = script_aligned["mean_H_baseline"].to_numpy(dtype=float)
    script_control_values = script_aligned[f"mean_H_{control_suffix}"].to_numpy(dtype=float)

    morph_abs_errors = abs(morph_control_values - morph_baseline_values)
    script_abs_errors = abs(script_control_values - script_baseline_values)

    joint_baseline_values = pd.Series([*morph_baseline_values, *script_baseline_values], dtype=float).to_numpy()
    joint_control_values = pd.Series([*morph_control_values, *script_control_values], dtype=float).to_numpy()
    joint_abs_errors = pd.Series([*morph_abs_errors, *script_abs_errors], dtype=float)

    return {
        "n_dims_morph": int(len(morph_aligned)),
        "n_dims_script": int(len(script_aligned)),
        "morph_mean_mae": float(morph_abs_errors.mean()),
        "script_mean_mae": float(script_abs_errors.mean()),
        "joint_mean_mae": float(joint_abs_errors.mean()),
        "morph_mean_r2": compute_curve_r2(morph_baseline_values, morph_control_values),
        "script_mean_r2": compute_curve_r2(script_baseline_values, script_control_values),
        "joint_mean_r2": compute_curve_r2(joint_baseline_values, joint_control_values),
    }


def print_metric_stats(label, stats):
    print(
        f"{label}: "
        f"morph_mean_mae={stats['morph_mean_mae']:.6g}, "
        f"script_mean_mae={stats['script_mean_mae']:.6g}, "
        f"joint_mean_mae={stats['joint_mean_mae']:.6g}, "
        f"morph_mean_r2={stats['morph_mean_r2']:.6g}, "
        f"script_mean_r2={stats['script_mean_r2']:.6g}, "
        f"joint_mean_r2={stats['joint_mean_r2']:.6g}"
    )


def summarize_seed_metrics(metric_df, *, seed_col, model_col):
    if metric_df.empty:
        return pd.DataFrame()

    seed_df = metric_df.loc[metric_df["result_type"] == "seed"].copy()
    if seed_df.empty:
        return pd.DataFrame()

    value_cols = [
        "morph_mean_mae",
        "script_mean_mae",
        "joint_mean_mae",
        "morph_mean_r2",
        "script_mean_r2",
        "joint_mean_r2",
    ]
    grouped = seed_df.groupby(model_col, as_index=False)
    summary_df = grouped.agg(
        n_seeds=(seed_col, "count"),
        morph_mean_mae_mean=("morph_mean_mae", "mean"),
        morph_mean_mae_std=("morph_mean_mae", "std"),
        script_mean_mae_mean=("script_mean_mae", "mean"),
        script_mean_mae_std=("script_mean_mae", "std"),
        joint_mean_mae_mean=("joint_mean_mae", "mean"),
        joint_mean_mae_std=("joint_mean_mae", "std"),
        morph_mean_r2_mean=("morph_mean_r2", "mean"),
        morph_mean_r2_std=("morph_mean_r2", "std"),
        script_mean_r2_mean=("script_mean_r2", "mean"),
        script_mean_r2_std=("script_mean_r2", "std"),
        joint_mean_r2_mean=("joint_mean_r2", "mean"),
        joint_mean_r2_std=("joint_mean_r2", "std"),
    )
    return summary_df


## Random PCA vs Full

In [4]:
def compute_random_pca_mae_r2_report():
    rows = []

    for model_config in MODEL_RUN_CONFIGS:
        model_name = model_config["model_name"]
        baseline_model_dir = Path(BASELINE_OUT_ROOT) / model_name / SPACE_NAME
        random_dir = baseline_model_dir / "random"

        baseline_morph_path = baseline_model_dir / "kondrak_global_summary.csv"
        baseline_script_path = baseline_model_dir / "script_entropy_summary.csv"
        random_morph_all_path = random_dir / "kondrak_global_summary_all_seeds.csv"
        random_script_all_path = random_dir / "script_entropy_all_seeds.csv"

        required_paths = [
            baseline_morph_path,
            baseline_script_path,
            random_morph_all_path,
            random_script_all_path,
        ]
        paths_exist, missing_paths = require_existing_paths(required_paths)
        print()
        print(f"=== {model_name} | random PCA ===")
        if not paths_exist:
            print(f"{model_name}: missing files")
            for missing_path in missing_paths:
                print(f"  {missing_path}")
            continue

        baseline_morph = pd.read_csv(baseline_morph_path)
        baseline_script = pd.read_csv(baseline_script_path)
        random_morph_all = pd.read_csv(random_morph_all_path)
        random_script_all = pd.read_csv(random_script_all_path)

        for frame_name, frame in [
            ("random_morph_all", random_morph_all),
            ("random_script_all", random_script_all),
        ]:
            if "pca_seed" not in frame.columns:
                raise ValueError(f"{frame_name} missing pca_seed column")

        random_morph_mean = (
            random_morph_all
            .groupby("pca_dim", as_index=False)["global_mean"]
            .mean()
        )
        random_script_mean = (
            random_script_all
            .groupby("pca_dim", as_index=False)["mean_H"]
            .mean()
        )
        mean_stats = compute_morph_script_joint_stats(
            baseline_morph,
            baseline_script,
            random_morph_mean,
            random_script_mean,
            control_suffix="control",
        )
        print_metric_stats("mean_curve", mean_stats)
        rows.append({
            "baseline_model_name": model_name,
            "control_model_name": model_name,
            "control_type": "random_pca",
            "result_type": "mean_curve",
            "pca_seed": pd.NA,
            "perm_seed": pd.NA,
            **mean_stats,
        })

        common_seeds = sorted(
            set(random_morph_all["pca_seed"].astype(int)).intersection(
                set(random_script_all["pca_seed"].astype(int))
            )
        )
        for pca_seed in common_seeds:
            seed_morph = random_morph_all.loc[random_morph_all["pca_seed"].astype(int) == int(pca_seed)].copy()
            seed_script = random_script_all.loc[random_script_all["pca_seed"].astype(int) == int(pca_seed)].copy()
            seed_stats = compute_morph_script_joint_stats(
                baseline_morph,
                baseline_script,
                seed_morph,
                seed_script,
                control_suffix="control",
            )
            print_metric_stats(f"seed={int(pca_seed)}", seed_stats)
            rows.append({
                "baseline_model_name": model_name,
                "control_model_name": model_name,
                "control_type": "random_pca",
                "result_type": "seed",
                "pca_seed": int(pca_seed),
                "perm_seed": pd.NA,
                **seed_stats,
            })

    return pd.DataFrame(rows)


In [5]:
random_pca_mae_r2_df = compute_random_pca_mae_r2_report()
random_pca_mae_r2_df



=== mistralai/Mistral-7B-v0.1 | random PCA ===
mean_curve: morph_mean_mae=0.0038249, script_mean_mae=0.00238346, joint_mean_mae=0.00310418, morph_mean_r2=0.997197, script_mean_r2=0.996615, joint_mean_r2=0.999344
seed=0: morph_mean_mae=0.00405658, script_mean_mae=0.00433152, joint_mean_mae=0.00419405, morph_mean_r2=0.997455, script_mean_r2=0.990066, joint_mean_r2=0.999064
seed=42: morph_mean_mae=0.00392932, script_mean_mae=0.00699476, joint_mean_mae=0.00546204, morph_mean_r2=0.996318, script_mean_r2=0.965681, joint_mean_r2=0.997657
seed=1000: morph_mean_mae=0.00561598, script_mean_mae=0.00414232, joint_mean_mae=0.00487915, morph_mean_r2=0.994095, script_mean_r2=0.984095, joint_mean_r2=0.998184
seed=9999: morph_mean_mae=0.00451623, script_mean_mae=0.00484446, joint_mean_mae=0.00468035, morph_mean_r2=0.995349, script_mean_r2=0.978528, joint_mean_r2=0.998126
seed=827307999: morph_mean_mae=0.00330184, script_mean_mae=0.00378794, joint_mean_mae=0.00354489, morph_mean_r2=0.997783, script_mea

,baseline_model_name,control_model_name,control_type,result_type,pca_seed,perm_seed,n_dims_morph,n_dims_script,morph_mean_mae,script_mean_mae,joint_mean_mae,morph_mean_r2,script_mean_r2,joint_mean_r2
0,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,mean_curve,<NA>,<NA>,9,9,0.003825,0.002383,0.003104,0.997197,0.996615,0.999344
1,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,0,<NA>,9,9,0.004057,0.004332,0.004194,0.997455,0.990066,0.999064
2,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,42,<NA>,9,9,0.003929,0.006995,0.005462,0.996318,0.965681,0.997657
3,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1000,<NA>,9,9,0.005616,0.004142,0.004879,0.994095,0.984095,0.998184
4,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,9999,<NA>,9,9,0.004516,0.004844,0.004680,0.995349,0.978528,0.998126
5,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,827307999,<NA>,9,9,0.003302,0.003788,0.003545,0.997783,0.991733,0.999204
6,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1627694678,<NA>,9,9,0.003820,0.001797,0.002809,0.997446,0.998969,0.999504
7,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1813382118,<NA>,9,9,0.002875,0.004875,0.003875,0.998475,0.983470,0.998915
8,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1911784257,<NA>,9,9,0.005113,0.004460,0.004787,0.991732,0.988060,0.997969
9,mistralai/Mixtral-8x7B-v0.1,mistralai/Mixtral-8x7B-v0.1,random_pca,mean_curve,<NA>,<NA>,9,9,0.001705,0.001180,0.001443,0.999465,0.998130,0.999863


## Random PCA Seed Summary

In [6]:
random_pca_seed_summary_df = summarize_seed_metrics(
    random_pca_mae_r2_df,
    seed_col="pca_seed",
    model_col="baseline_model_name",
)
random_pca_seed_summary_df


,baseline_model_name,n_seeds,morph_mean_mae_mean,morph_mean_mae_std,script_mean_mae_mean,script_mean_mae_std,joint_mean_mae_mean,joint_mean_mae_std,morph_mean_r2_mean,morph_mean_r2_std,script_mean_r2_mean,script_mean_r2_std,joint_mean_r2_mean,joint_mean_r2_std
0,gpt-oss,8,0.002796,0.001041,0.004238,0.001467,0.003517,0.000820,0.997261,0.002693,0.993072,0.005750,0.998397,0.001023
1,mistralai/Mistral-7B-v0.1,8,0.004154,0.000904,0.004404,0.001433,0.004279,0.000849,0.996082,0.002262,0.985075,0.009955,0.998578,0.000674
2,mistralai/Mixtral-8x7B-v0.1,8,0.002503,0.001259,0.002576,0.001222,0.002539,0.000711,0.998027,0.002260,0.981093,0.020083,0.999276,0.000464


## Permutation vs Full

In [7]:
def compute_permutation_mae_r2_report():
    rows = []

    for model_config in PERMUTATION_MODEL_CONFIGS:
        baseline_model_name = model_config["baseline_model_name"]
        permutation_model_name = model_config["permutation_model_name"]
        baseline_model_dir = Path(BASELINE_OUT_ROOT) / baseline_model_name / SPACE_NAME
        permutation_dir = Path(BASELINE_OUT_ROOT) / permutation_model_name / SPACE_NAME

        baseline_morph_path = baseline_model_dir / "kondrak_global_summary.csv"
        baseline_script_path = baseline_model_dir / "script_entropy_summary.csv"
        permutation_morph_mean_path = permutation_dir / "kondrak_global_summary.csv"
        permutation_script_mean_path = permutation_dir / "script_entropy_summary.csv"
        permutation_morph_all_path = permutation_dir / "kondrak_global_summary_all_seeds.csv"
        permutation_script_all_path = permutation_dir / "script_entropy_summary_all_seeds.csv"

        required_paths = [
            baseline_morph_path,
            baseline_script_path,
            permutation_morph_mean_path,
            permutation_script_mean_path,
            permutation_morph_all_path,
            permutation_script_all_path,
        ]
        paths_exist, missing_paths = require_existing_paths(required_paths)
        print()
        print(f"=== {baseline_model_name} | permutation ===")
        if not paths_exist:
            print(f"{baseline_model_name}: missing files")
            for missing_path in missing_paths:
                print(f"  {missing_path}")
            continue

        baseline_morph = pd.read_csv(baseline_morph_path)
        baseline_script = pd.read_csv(baseline_script_path)
        permutation_morph_mean = pd.read_csv(permutation_morph_mean_path)
        permutation_script_mean = pd.read_csv(permutation_script_mean_path)
        permutation_morph_all = pd.read_csv(permutation_morph_all_path)
        permutation_script_all = pd.read_csv(permutation_script_all_path)

        mean_stats = compute_morph_script_joint_stats(
            baseline_morph,
            baseline_script,
            permutation_morph_mean,
            permutation_script_mean,
            control_suffix="control",
        )
        print_metric_stats("mean", mean_stats)
        rows.append({
            "baseline_model_name": baseline_model_name,
            "control_model_name": permutation_model_name,
            "control_type": "permutation",
            "result_type": "mean",
            "pca_seed": pd.NA,
            "perm_seed": pd.NA,
            **mean_stats,
        })

        for frame_name, frame in [
            ("permutation_morph_all", permutation_morph_all),
            ("permutation_script_all", permutation_script_all),
        ]:
            if "perm_seed" not in frame.columns:
                raise ValueError(f"{frame_name} missing perm_seed column")

        common_seeds = sorted(
            set(permutation_morph_all["perm_seed"].astype(int)).intersection(
                set(permutation_script_all["perm_seed"].astype(int))
            )
        )
        for perm_seed in common_seeds:
            seed_morph = permutation_morph_all.loc[permutation_morph_all["perm_seed"].astype(int) == int(perm_seed)].copy()
            seed_script = permutation_script_all.loc[permutation_script_all["perm_seed"].astype(int) == int(perm_seed)].copy()
            seed_stats = compute_morph_script_joint_stats(
                baseline_morph,
                baseline_script,
                seed_morph,
                seed_script,
                control_suffix="control",
            )
            print_metric_stats(f"seed={int(perm_seed)}", seed_stats)
            rows.append({
                "baseline_model_name": baseline_model_name,
                "control_model_name": permutation_model_name,
                "control_type": "permutation",
                "result_type": "seed",
                "pca_seed": pd.NA,
                "perm_seed": int(perm_seed),
                **seed_stats,
            })

    return pd.DataFrame(rows)


In [8]:
permutation_mae_r2_df = compute_permutation_mae_r2_report()
permutation_mae_r2_df



=== mistralai/Mistral-7B-v0.1 | permutation ===
mean: morph_mean_mae=0.358717, script_mean_mae=0.198126, joint_mean_mae=0.278421, morph_mean_r2=-9.13569, script_mean_r2=-15.0359, joint_mean_r2=-1.55892
seed=86939546: morph_mean_mae=0.357058, script_mean_mae=0.19071, joint_mean_mae=0.273884, morph_mean_r2=-9.05845, script_mean_r2=-12.857, joint_mean_r2=-1.4374
seed=556019485: morph_mean_mae=0.358695, script_mean_mae=0.196556, joint_mean_mae=0.277625, morph_mean_r2=-9.13191, script_mean_r2=-14.429, joint_mean_r2=-1.52816
seed=827307999: morph_mean_mae=0.359225, script_mean_mae=0.204991, joint_mean_mae=0.282108, morph_mean_r2=-9.16205, script_mean_r2=-16.5391, joint_mean_r2=-1.63807
seed=903170602: morph_mean_mae=0.35957, script_mean_mae=0.201297, joint_mean_mae=0.280433, morph_mean_r2=-9.17695, script_mean_r2=-16.0701, joint_mean_r2=-1.6174
seed=1043521778: morph_mean_mae=0.358137, script_mean_mae=0.19525, joint_mean_mae=0.276694, morph_mean_r2=-9.10428, script_mean_r2=-14.3427, joint_m

,baseline_model_name,control_model_name,control_type,result_type,pca_seed,perm_seed,n_dims_morph,n_dims_script,morph_mean_mae,script_mean_mae,joint_mean_mae,morph_mean_r2,script_mean_r2,joint_mean_r2
0,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,mean,<NA>,<NA>,9,9,0.358717,0.198126,0.278421,-9.135688,-15.035941,-1.558924
1,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,86939546,9,9,0.357058,0.190710,0.273884,-9.058447,-12.857003,-1.437401
2,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,556019485,9,9,0.358695,0.196556,0.277625,-9.131912,-14.428976,-1.528159
3,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,827307999,9,9,0.359225,0.204991,0.282108,-9.162054,-16.539100,-1.638073
4,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,903170602,9,9,0.359570,0.201297,0.280433,-9.176951,-16.070076,-1.617400
5,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,1043521778,9,9,0.358137,0.195250,0.276694,-9.104284,-14.342731,-1.519074
6,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,1097954097,9,9,0.359388,0.198491,0.278940,-9.153455,-15.175972,-1.568961
7,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,1627694678,9,9,0.359378,0.198664,0.279021,-9.174429,-15.236672,-1.575621
8,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,1813382118,9,9,0.357290,0.198426,0.277858,-9.082793,-15.306698,-1.563151
9,mistralai/Mistral-7B-v0.1,permutation/mistralai/Mistral-7B-v0.1_rand,permutation,seed,<NA>,1911784257,9,9,0.359496,0.197647,0.278572,-9.178169,-15.148689,-1.571908


## Permutation Seed Summary

In [9]:
permutation_seed_summary_df = summarize_seed_metrics(
    permutation_mae_r2_df,
    seed_col="perm_seed",
    model_col="baseline_model_name",
)
permutation_seed_summary_df


,baseline_model_name,n_seeds,morph_mean_mae_mean,morph_mean_mae_std,script_mean_mae_mean,script_mean_mae_std,joint_mean_mae_mean,joint_mean_mae_std,morph_mean_r2_mean,morph_mean_r2_std,script_mean_r2_mean,script_mean_r2_std,joint_mean_r2_mean,joint_mean_r2_std
0,gpt-oss,10,0.396054,0.000093,0.158325,0.001013,0.277190,0.000523,-16.150625,0.006190,-2.943925,0.060158,-2.799808,0.009732
1,mistralai/Mistral-7B-v0.1,10,0.358717,0.000922,0.198126,0.003725,0.278421,0.002192,-9.136251,0.041898,-15.065326,1.017866,-1.560479,0.055987
2,mistralai/Mixtral-8x7B-v0.1,10,0.338218,0.000722,0.167660,0.003004,0.252939,0.001650,-8.808612,0.025330,-36.971735,1.440481,-1.575102,0.028782


## Combined Results

In [10]:
combined_mae_r2_df = pd.concat(
    [random_pca_mae_r2_df, permutation_mae_r2_df],
    ignore_index=True,
)
combined_mae_r2_df


,baseline_model_name,control_model_name,control_type,result_type,pca_seed,perm_seed,n_dims_morph,n_dims_script,morph_mean_mae,script_mean_mae,joint_mean_mae,morph_mean_r2,script_mean_r2,joint_mean_r2
0,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,mean_curve,<NA>,<NA>,9,9,0.003825,0.002383,0.003104,0.997197,0.996615,0.999344
1,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,0,<NA>,9,9,0.004057,0.004332,0.004194,0.997455,0.990066,0.999064
2,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,42,<NA>,9,9,0.003929,0.006995,0.005462,0.996318,0.965681,0.997657
3,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1000,<NA>,9,9,0.005616,0.004142,0.004879,0.994095,0.984095,0.998184
4,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,9999,<NA>,9,9,0.004516,0.004844,0.004680,0.995349,0.978528,0.998126
5,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,827307999,<NA>,9,9,0.003302,0.003788,0.003545,0.997783,0.991733,0.999204
6,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1627694678,<NA>,9,9,0.003820,0.001797,0.002809,0.997446,0.998969,0.999504
7,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1813382118,<NA>,9,9,0.002875,0.004875,0.003875,0.998475,0.983470,0.998915
8,mistralai/Mistral-7B-v0.1,mistralai/Mistral-7B-v0.1,random_pca,seed,1911784257,<NA>,9,9,0.005113,0.004460,0.004787,0.991732,0.988060,0.997969
9,mistralai/Mixtral-8x7B-v0.1,mistralai/Mixtral-8x7B-v0.1,random_pca,mean_curve,<NA>,<NA>,9,9,0.001705,0.001180,0.001443,0.999465,0.998130,0.999863


# Random Orthogonal Projection Macro-Trend Statistics

In [4]:
def compute_random_projection_mae_r2_report():
    rows = []

    for model_config in MODEL_RUN_CONFIGS:
        model_name = model_config["model_name"]
        model_dir = Path(BASELINE_OUT_ROOT) / model_name / SPACE_NAME
        random_projection_dir = model_dir / "random_projection"
        required_paths = [
            model_dir / "kondrak_global_summary.csv",
            model_dir / "script_entropy_summary.csv",
            random_projection_dir / "kondrak_global_summary_mean.csv",
            random_projection_dir / "script_entropy_summary_mean.csv",
            random_projection_dir / "kondrak_global_summary_all_seeds.csv",
            random_projection_dir / "script_entropy_summary_all_seeds.csv",
        ]
        paths_exist, missing_paths = require_existing_paths(required_paths)
        print()
        print(f"=== {model_name} | random projection ===")
        if not paths_exist:
            for missing_path in missing_paths:
                print(f"  {missing_path}")
            continue

        baseline_morph = pd.read_csv(required_paths[0])
        baseline_script = pd.read_csv(required_paths[1])
        rp_morph_mean = pd.read_csv(required_paths[2])
        rp_script_mean = pd.read_csv(required_paths[3])
        rp_morph_all = pd.read_csv(required_paths[4])
        rp_script_all = pd.read_csv(required_paths[5])

        mean_stats = compute_morph_script_joint_stats(
            baseline_morph,
            baseline_script,
            rp_morph_mean,
            rp_script_mean,
            control_suffix="control",
        )
        print_metric_stats("mean_curve", mean_stats)
        rows.append({
            "baseline_model_name": model_name,
            "control_model_name": f"{model_name}/random_projection",
            "control_type": "random_projection",
            "result_type": "mean_curve",
            "pca_seed": pd.NA,
            "perm_seed": pd.NA,
            "rp_seed": pd.NA,
            **mean_stats,
        })

        common_seeds = sorted(
            set(rp_morph_all["rp_seed"].astype(int)).intersection(
                set(rp_script_all["rp_seed"].astype(int))
            )
        )
        for rp_seed in common_seeds:
            seed_morph = rp_morph_all.loc[
                rp_morph_all["rp_seed"].astype(int) == int(rp_seed)
            ].copy()
            seed_script = rp_script_all.loc[
                rp_script_all["rp_seed"].astype(int) == int(rp_seed)
            ].copy()
            seed_stats = compute_morph_script_joint_stats(
                baseline_morph,
                baseline_script,
                seed_morph,
                seed_script,
                control_suffix="control",
            )
            print_metric_stats(f"seed={int(rp_seed)}", seed_stats)
            rows.append({
                "baseline_model_name": model_name,
                "control_model_name": f"{model_name}/random_projection",
                "control_type": "random_projection",
                "result_type": "seed",
                "pca_seed": pd.NA,
                "perm_seed": pd.NA,
                "rp_seed": int(rp_seed),
                **seed_stats,
            })

    return pd.DataFrame(rows)


random_projection_mae_r2_df = compute_random_projection_mae_r2_report()
random_projection_seed_summary_df = summarize_seed_metrics(
    random_projection_mae_r2_df,
    seed_col="rp_seed",
    model_col="baseline_model_name",
)
combined_mae_r2_with_random_projection_df = pd.concat(
    [combined_mae_r2_df, random_projection_mae_r2_df],
    ignore_index=True,
)
random_projection_mae_r2_df.to_csv(
    Path(BASELINE_OUT_ROOT) / "random_projection_mae_r2.csv",
    index=False,
)
random_projection_seed_summary_df


=== mistralai/Mistral-7B-v0.1 | random projection ===
mean_curve: morph_mean_mae=0.0269879, script_mean_mae=0.02458, joint_mean_mae=0.0257839, morph_mean_r2=0.810231, script_mean_r2=-0.0465514, joint_mean_r2=0.91507
seed=86939546: morph_mean_mae=0.0264345, script_mean_mae=0.0229316, joint_mean_mae=0.0246831, morph_mean_r2=0.810874, script_mean_r2=0.0576906, joint_mean_r2=0.920352
seed=556019485: morph_mean_mae=0.0266588, script_mean_mae=0.0249765, joint_mean_mae=0.0258177, morph_mean_r2=0.815009, script_mean_r2=-0.105855, joint_mean_r2=0.912959
seed=827307999: morph_mean_mae=0.0263692, script_mean_mae=0.0256218, joint_mean_mae=0.0259955, morph_mean_r2=0.81434, script_mean_r2=-0.125543, joint_mean_r2=0.911866
seed=903170602: morph_mean_mae=0.0275295, script_mean_mae=0.0253224, joint_mean_mae=0.0264259, morph_mean_r2=0.796828, script_mean_r2=-0.114018, joint_mean_r2=0.909391
seed=1043521778: morph_mean_mae=0.0276953, script_mean_mae=0.0258168, joint_mean_mae=0.0267561, morph_mean_r2=0.8

NameError: name 'combined_mae_r2_df' is not defined

In [5]:
random_projection_cluster_stat_frames = []
for model_config in MODEL_RUN_CONFIGS:
    model_name = model_config["model_name"]
    summary_path = (
        Path(BASELINE_OUT_ROOT)
        / model_name
        / SPACE_NAME
        / "random_projection"
        / "summary_all_seeds.csv"
    )
    summary_df = pd.read_csv(summary_path)
    summary_df["model_name"] = model_name
    random_projection_cluster_stat_frames.append(summary_df)

random_projection_cluster_all_df = pd.concat(
    random_projection_cluster_stat_frames,
    axis=0,
    ignore_index=True,
)
random_projection_cluster_statistics_df = (
    random_projection_cluster_all_df
    .groupby(["model_name", "pca_dim"], as_index=False)
    .agg(
        n_seeds=("rp_seed", "nunique"),
        cluster_count_mean=("n_clusters_excl_noise", "mean"),
        cluster_count_std=("n_clusters_excl_noise", "std"),
        noise_fraction_mean=("noise_ratio", "mean"),
        noise_fraction_std=("noise_ratio", "std"),
        mean_cluster_size_mean=("mean_cluster_size", "mean"),
        mean_cluster_size_std=("mean_cluster_size", "std"),
        median_cluster_size_mean=("median_cluster_size", "mean"),
        median_cluster_size_std=("median_cluster_size", "std"),
    )
)
random_projection_cluster_statistics_df.to_csv(
    Path(BASELINE_OUT_ROOT) / "random_projection_cluster_statistics.csv",
    index=False,
)
random_projection_cluster_statistics_df

,model_name,pca_dim,n_seeds,cluster_count_mean,cluster_count_std,noise_fraction_mean,noise_fraction_std,mean_cluster_size_mean,mean_cluster_size_std,median_cluster_size_mean,median_cluster_size_std
0,gpt-oss,6,10,3259.1,1144.689327,0.734596,0.196015,8287.417700,26171.390197,8284.45,26172.432941
1,gpt-oss,182,10,3345.1,38.338985,0.701146,0.003386,17.967941,0.317238,12.70,0.483046
2,gpt-oss,466,10,3680.5,13.100042,0.669124,0.001315,18.078038,0.114998,13.00,0.000000
3,gpt-oss,739,10,3767.6,24.604426,0.662117,0.001772,18.034620,0.161592,12.95,0.158114
4,gpt-oss,1591,10,3859.1,19.075581,0.657397,0.000462,17.852611,0.096799,13.00,0.000000
5,gpt-oss,2264,10,3863.6,21.459005,0.656061,0.001710,17.901453,0.138512,12.90,0.316228
6,gpt-oss,2532,10,3863.1,16.030873,0.654868,0.002820,17.965830,0.188953,13.00,0.000000
7,gpt-oss,2868,10,3875.4,8.707596,0.657300,0.000813,17.782266,0.072204,13.00,0.000000
8,gpt-oss,2880,10,3877.3,0.483046,0.657827,0.000028,17.746086,0.002298,13.00,0.000000
9,mistralai/Mistral-7B-v0.1,5,10,721.8,14.868311,0.727078,0.004413,12.103368,0.277757,9.00,0.000000
